# EAGLE-3 draft for granite-docling

The Medusa route works but loses: `medusa` is in neither `EagleModelTypes` nor the V2-runner
allowlist, so vLLM disables async scheduling *and* falls back to the V1 model runner. Measured, a
drafted round costs 3.33 ms against plain vLLM's 1.20 ms, and break-even lands at 2.78 tokens/step
where the head delivers 1.20 pooled.

`eagle3` keeps both. Measured here with a random-init drafter (acceptance 0, so tok/s *is* the
round cost):

| config | round | drafter adds | break-even |
|---|---:|---:|---:|
| plain vLLM | 1.352 ms | — | 1.000 |
| eagle3 k=1 | 2.147 ms | 0.795 ms | **1.588** |
| eagle3 k=2 | 2.333 ms | 0.981 ms | **1.726** |
| eagle3 k=3 | 2.442 ms | 1.090 ms | 1.807 |
| eagle3 k=5 | 2.970 ms | 1.618 ms | 2.197 |

So eagle3 roughly halves Medusa's bar (2.78 → 1.59 at k=1) but it is **not** the ~1.1 that a
free-drafter calculation suggests: the drafter has a ~0.68 ms fixed cost plus ~0.11 ms per extra
step, and that is paid on every round. Low k is favoured — the marginal token is cheap but the
first one is not.

**What this model is.** Not the latent draft ported over. EAGLE-3 is autoregressive: propose a
token, embed it, feed it back through the drafter's own KV cache, repeat. `Eagle3LlamaForCausalLM`
returns one hidden state per position, so the `[B, T, HORIZON, 576]` parallel-head contract does
not survive — `future_heads` has nowhere to go. What *does* survive is the window: the drafter
attends over its own KV cache, so history comes back without being passed in.

**Prerequisite: traces with EAGLE-3 taps.** The V2 runner sets `use_aux_hidden_state_outputs=True`
unconditionally for `method="eagle3"`, so the target emits layers 2/14/27 concatenated and the
drafter's `fc` maps 1728 → 576. The current corpus stores only `last_hidden_state`, so it must be
re-extracted (4x larger on disk):

```
uv run fastdocling-extract data/images data/traces_taps --keep-taps
```


In [ ]:
import json
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch.nn import functional as F
from torchinfo import summary
from tqdm.auto import tqdm

from fastdocling.data import FIRST_OFFSET, corpus_key, iterate_batches, output_vocab, plan, scan_traces, split
from fastdocling.eagle3 import AUX_LAYERS, Eagle3Draft, export_eagle3_checkpoint
from fastdocling.target import MODEL_ID, load_lm_head

# Paths resolve against the repo root, not the working directory: this notebook lives in
# notebooks/eagle3/, so a bare Path("data/traces") would point at notebooks/eagle3/data/traces.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
TRACES = ROOT / "data/traces_taps"   # MUST be a --keep-taps extraction; see the note above
CACHE = ROOT / "data/cache"
OUTPUT = ROOT / "checkpoints/eagle3_draft.pt"
DECODES = CACHE / "decodes"

CONTEXT_LENGTH = 64     # training window; at inference the drafter's own KV cache carries history
BATCH_SIZE = 48
EPOCHS = 3
SPEC_TOKENS = 2         # num_speculative_tokens; k=1..2 is where break-even is lowest
MUON_LR, ADAMW_LR, WARMUP_STEPS = 0.02, 3e-4, 10
VLLM_GPU_FRACTION = 0.35
IN_DOMAIN_PAGES = 8     # holdout pages measured end to end; 3 lets a single page dominate
                        # the pooled ratio, which is how 1.20 tokens/step once read as 2.07
PRUNE_VOCAB = True      # d2t pruning; unlike Medusa's token_map this is not broken upstream

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
infos = scan_traces(TRACES, min_completion=CONTEXT_LENGTH + 2)
missing = [i for i in infos if not i.has_taps]
if missing:
    raise RuntimeError(
        f"{len(missing)}/{len(infos)} traces in {TRACES} have no EAGLE-3 taps.  vLLM's V2 runner "
        f"always requests aux hidden states for method='eagle3', so the drafter's fc needs "
        f"layers {AUX_LAYERS} concatenated (1728-d).  Re-extract with:\n"
        f"  uv run fastdocling-extract data/images {TRACES.name} --keep-taps")

train_infos, holdout_infos = split(infos, holdout_fraction=0.02)
schedule = plan(train_infos, CONTEXT_LENGTH, BATCH_SIZE, horizon=2)
TOTAL_STEPS = schedule.steps_for_epochs(EPOCHS)
CORPUS_KEY = corpus_key(train_infos)
print(schedule.describe(steps=TOTAL_STEPS))
print(f"holdout: {len(holdout_infos)} pages; corpus key {CORPUS_KEY}")

vocab = output_vocab(train_infos, cache=TRACES / f"output_vocab_{CORPUS_KEY}.npy") if PRUNE_VOCAB else None
if vocab is not None:
    print(f"draft vocabulary pruned to {len(vocab):,} of 100,352 tokens (written as d2t)")

2,695 pages, 2,925,395 usable DocTags positions
context=64 horizon=2 batch=32 -> 2,048 tokens/step
42,333 windows/epoch -> 1,322 steps/epoch
1,322 steps = 1.00 epochs
holdout: 55 pages; corpus key 08eb21b227
draft vocabulary pruned to 30,753 of 100,352 tokens (written as d2t)


## Alignment

At cursor `i` the target's state `h_i` is known, and so is token `i+1` — it is the bonus token the
target already emitted. The drafter embeds that token, combines it with `h_i`, and must predict
token `i+2`. So the drafter's *input* token is at offset 1 and its *label* is at offset 2, which is
`FIRST_OFFSET`, the same convention the latent draft and the live loop use.

`iterate_batches(..., horizon=2, first_offset=1)` yields exactly those two columns:
`tokens[:, :, 0]` is the input embedding's token, `tokens[:, :, 1]` is the label.

In [3]:
draft = Eagle3Draft(
    in_dim=576 * len(AUX_LAYERS),
    draft_vocab_size=len(vocab) if vocab is not None else 100_352,
).to(device)

# The target's embedding and vocabulary projection are frozen and shared, so they are not trained
# here and are omitted from the export -- vLLM binds the target's own tensors instead.  What is
# left is fc + one decoder layer + norm: the part that actually has to be learned.
trainable = [p for n, p in draft.named_parameters() if not n.startswith(("embed_tokens", "lm_head"))]
for n, p in draft.named_parameters():
    if n.startswith(("embed_tokens", "lm_head")):
        p.requires_grad_(False)
print(f"{sum(p.numel() for p in trainable):,} trainable of {sum(p.numel() for p in draft.parameters()):,} total")

summary(draft, input_data=(torch.zeros(2, CONTEXT_LENGTH, 576 * len(AUX_LAYERS), device=device),
                           torch.zeros(2, CONTEXT_LENGTH, dtype=torch.long, device=device)),
        depth=3, col_names=("input_size", "output_size", "num_params"), row_settings=("var_names",))

5,089,536 trainable of 80,606,016 total


Layer (type (var_name))                       Input Shape               Output Shape              Param #
Eagle3Draft (Eagle3Draft)                     [2, 64, 1728]             [2, 64, 30753]            --
├─Linear (fc)                                 [2, 64, 1728]             [2, 64, 576]              995,328
├─Embedding (embed_tokens)                    [2, 64]                   [2, 64, 576]              (57,802,752)
├─Eagle3Layer (layer)                         [2, 64, 576]              [2, 64, 576]              --
│    └─RMSNorm (input_layernorm)              [2, 64, 576]              [2, 64, 576]              576
│    └─RMSNorm (hidden_norm)                  [2, 64, 576]              [2, 64, 576]              576
│    └─Linear (q_proj)                        [2, 64, 1152]             [2, 64, 576]              663,552
│    └─Linear (k_proj)                        [2, 64, 1152]             [2, 64, 192]              221,184
│    └─Linear (v_proj)                        [2, 64, 1152]

In [4]:
muon_params = [p for p in trainable if p.ndim == 2]
adamw_params = [p for p in trainable if p.ndim != 2]
optimizers = [torch.optim.Muon(muon_params, lr=MUON_LR, weight_decay=0.01, momentum=0.95, nesterov=True)]
if adamw_params:
    optimizers.append(torch.optim.AdamW(adamw_params, lr=ADAMW_LR, weight_decay=0.01))
warmup_cosine = lambda s: min(1.0, (s + 1) / WARMUP_STEPS) * 0.5 * (1 + np.cos(np.pi * min(1.0, s / max(1, TOTAL_STEPS))))
lr_schedules = [torch.optim.lr_scheduler.LambdaLR(o, warmup_cosine) for o in optimizers]

# Map target token ids onto draft rows when the head is pruned; ids outside the set are ignored.
if vocab is not None:
    target_to_draft = torch.full((100_352,), -100, dtype=torch.long, device=device)
    target_to_draft[torch.as_tensor(vocab, device=device)] = torch.arange(len(vocab), device=device)
else:
    target_to_draft = None


def batches(infos, seed=0, drop_last=True):
    """(aux_states [B,T,1728], input token [B,T], label [B,T]) -- see the alignment note."""
    for x, _, tok in iterate_batches(infos, CONTEXT_LENGTH, BATCH_SIZE, horizon=2, features="eagle3",
                                     seed=seed, device=device, drop_last=drop_last, first_offset=1):
        yield x, tok[:, :, 0], tok[:, :, 1]


@torch.inference_mode()
def acceptance(infos):
    """Teacher-forced top-1 accuracy of the drafter's next-token prediction."""
    draft.eval(); hits = total = 0
    for x, inp, label in batches(infos, seed=0, drop_last=False):
        pred = draft(x, inp).argmax(-1)
        gold = target_to_draft[label] if target_to_draft is not None else label
        mask = gold >= 0
        hits += (pred[mask] == gold[mask]).sum().item(); total += int(mask.sum())
    draft.train()
    return hits / max(1, total)


history, step, started = [], 0, perf_counter()
progress = tqdm(total=TOTAL_STEPS, unit="step", dynamic_ncols=True)
for epoch in range(EPOCHS):
    for x, inp, label in batches(train_infos, seed=epoch):
        gold = target_to_draft[label] if target_to_draft is not None else label
        loss = F.cross_entropy(draft(x, inp).flatten(0, 1), gold.flatten(), ignore_index=-100)
        for o in optimizers:
            o.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        for o, s in zip(optimizers, lr_schedules):
            o.step(); s.step()
        step += 1; progress.update()
        if step % 200 == 0:
            history.append((step, loss.item()))
            progress.set_postfix(loss=f"{loss.item():.3f}", lr=f"{lr_schedules[0].get_last_lr()[0]:.1e}")
progress.close()

p1 = acceptance(holdout_infos)
print(f"\nholdout next-token accuracy {p1:.3f}  ->  k=1 gives {1 + p1:.3f} tokens/step "
      f"(break-even 1.588), k=2 at best {1 + p1 + p1 ** 2:.3f} (break-even 1.726)")
OUTPUT.parent.mkdir(exist_ok=True)
torch.save({"state_dict": draft.state_dict(), "accuracy": p1, "aux_layers": list(AUX_LAYERS)}, OUTPUT)

  0%|          | 0/1322 [00:00<?, ?step/s]


holdout next-token accuracy 0.650  ->  k=1 gives 1.650 tokens/step (break-even 1.588), k=2 at best 2.073 (break-even 1.726)


## Measure it in vLLM

Export writes vLLM's on-disk EAGLE-3 layout (`layers.0.*`, `fc`, `norm`, optional `d2t`) and omits
`embed_tokens`/`lm_head` so the engine binds the target's own. Report **pooled** aggregates — total
tokens over total seconds, total tokens over total rounds. Per-page means were distorted by a
factor of two on this page set, because one short page with high acceptance dominates an unweighted
average.

In [5]:
from fastdocling.decode import available
from fastdocling.decode.vllm_decoder import sweep_speculative


def pooled(rows):
    """(tokens, tok/s, tokens per target step) summed over pages before dividing.

    Totals first, ratio second.  An unweighted mean of per-page rates lets a short page outvote
    a long one: on this corpus a 729-token page at 3.14 tokens/step and a 7050-token page at
    1.09 averaged to 2.07, against a pooled 1.20.
    """
    tok = sum(r["tokens"] for r in rows)
    sec = sum(r["tokens"] / r["decode_tps"] for r in rows)
    rounds = sum(r["tokens"] / r["tokens_per_round"] for r in rows)
    return tok, tok / sec, tok / rounds


def report(rows, title):
    """Pooled throughput per configuration, then what the drafter actually got right."""
    grouped = {}
    for r in rows:
        grouped.setdefault(r["config"], []).append(r)

    n_pages = len(next(iter(grouped.values())))
    print(f"\n=== {title}: {n_pages} pages, {pooled(grouped['baseline'])[0]:,} tokens ===")
    print(f"{'configuration':22s}{'tok/s (pooled)':>16s}{'tokens/step':>13s}{'vs plain':>10s}")
    plain = None
    for label, rs in grouped.items():
        _, tps, per_step = pooled(rs)
        plain = plain or tps
        print(f"{label:22s}{tps:16.1f}{per_step:13.3f}{tps / plain:9.2f}x")

    # accepted_per_pos[i] counts rounds whose proposal at slot i was accepted.  Slot i is only
    # reached when slot i-1 was accepted, so its denominator is the acceptances at slot i-1 (and
    # for slot 0, the number of drafting rounds).  A flat accepted/proposed hides that the later
    # slots are judged on a much smaller population -- and it is the later slots that decide
    # whether raising num_speculative_tokens pays, since break-even rises with k.
    spec_rows = [r for r in rows if r["spec"] is not None]
    if not spec_rows:
        return grouped
    width = max(len(r["accepted_per_pos"]) for r in spec_rows)
    per_pos = [sum(r["accepted_per_pos"][i] if i < len(r["accepted_per_pos"]) else 0
                   for r in spec_rows) for i in range(width)]
    rounds_drafted = sum(r["drafts"] for r in spec_rows)
    proposed = sum(r["draft_tokens"] for r in spec_rows)
    print(f"\ncorrectly drafted: {rounds_drafted:,} drafting rounds, {proposed:,} tokens proposed")
    print(f"{'slot':>6s}{'reached':>10s}{'accepted':>10s}{'rate':>8s}")
    reached = rounds_drafted
    for i, acc in enumerate(per_pos, 1):
        print(f"{i:>6d}{reached:>10,}{acc:>10,}{acc / max(1, reached):>8.1%}")
        reached = acc
    print(f"{'all':>6s}{proposed:>10,}{sum(per_pos):>10,}{sum(per_pos) / max(1, proposed):>8.1%}")
    return grouped


if "vllm" not in available():
    print("vLLM is not installed here (`uv sync --extra cuda`); skipping.")
else:
    ckpt_dir = export_eagle3_checkpoint(
        draft, CACHE / "eagle3_vllm", vocab=vocab, share_embeddings=vocab is None,
        aux_layers=AUX_LAYERS)
    print(f"exported {ckpt_dir}")

    image_for = lambda info: ROOT.joinpath("data/images", *info.path.stem.split("__")).with_suffix(".png")
    SPEC_CFG = {"method": "eagle3", "model": str(ckpt_dir), "num_speculative_tokens": SPEC_TOKENS}
    rows = sweep_speculative([image_for(i) for i in holdout_infos[:IN_DOMAIN_PAGES]],
                             [None, SPEC_CFG], cache_dir=DECODES, batched=False,
                             gpu_memory_utilization=VLLM_GPU_FRACTION)
    report(rows, "in-domain holdout (ML papers)")

W0914 19:01:36.442000 214362 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W0914 19:01:36.443000 214362 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W0914 19:01:36.444000 214362 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


exported /root/Code/fastdocling/data/cache/eagle3_vllm


[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


INFO 09-14 19:01:42 [api_utils.py:286] non-default args: {'max_model_len': 8192, 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.35, 'disable_log_stats': True, 'limit_mm_per_prompt': {'image': 1}, 'model': 'ibm-granite/granite-docling-258M'}
INFO 09-14 19:01:43 [model.py:684] Resolved architecture: Idefics3ForConditionalGeneration
INFO 09-14 19:01:43 [model.py:2021] Using max model len 8192
WARNING 09-14 19:01:43 [model.py:981] Model does not support mm_device_do_normalize, forcing mm_device_do_normalize = False.
INFO 09-14 19:01:44 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-14 19:01:44 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


W0914 19:01:49.216000 215710 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W0914 19:01:49.217000 215710 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W0914 19:01:49.217000 215710 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


...

(EngineCore pid=216096) INFO 09-14 19:03:27 [model_runner.py:960] Graph capturing finished in 3 secs, took 0.52 GiB
(EngineCore pid=216096) INFO 09-14 19:03:28 [gpu_worker.py:625] Available KV cache memory: 0.66 GiB
(EngineCore pid=216096) INFO 09-14 19:03:28 [gpu_worker.py:640] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.3500 is equivalent to --gpu-memory-utilization=0.3113 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.3887. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=216096) INFO 09-14 19:03:28 [kv_cache_utils.py:2032] GPU KV cache size: 29,840 tokens, Maximum concurrency for 8,192 tokens per request: 3.64x
(EngineCore pid=216096) INFO 09-14 19:03:28 [kernel_warmup.py:124] JIT kernel warmup starting.
(EngineCore pid=216096) INFO 09-14 19:03:28 [kernel_warmup.py:134] JIT kernel warmup finished in 0.00s.
(Engine

(EngineCore pid=216096) 2026-09-14 19:03:28,281 - INFO - autotuner.py:972 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=216096) 2026-09-14 19:03:28,427 - INFO - autotuner.py:995 - flashinfer.jit: [Autotuner]: Autotuning process ends
(EngineCore pid=216096) 2026-09-14 19:03:28,489 - INFO - autotuner.py:2652 - flashinfer.jit: [Autotuner]: Saved 0 configs to /root/.cache/vllm/flashinfer_autotune_cache/flashinfer/0.6.18/d0bc9f54d6dd9c9a33eac8276032bbc46b2695fdd80d77899fab9d8b7a86f665/autotune_configs.json (0 new, 0 from previous config)
Capturing prefill CUDA graphs (PIECEWISE):  43%|████▎     | 26/60 [00:00<00:00, 257.69it/s]

(EngineCore pid=216096) INFO 09-14 19:03:31 [speculator.py:151] Capturing model for speculator...


Capturing decode CUDA graphs (FULL): 100%|██████████| 38/38 [00:00<00:00, 277.46it/s]


(EngineCore pid=216096) INFO 09-14 19:03:32 [model_runner.py:960] Graph capturing finished in 4 secs, took 0.31 GiB
(EngineCore pid=216096) INFO 09-14 19:03:32 [gpu_worker.py:797] CUDA graph pool memory: 0.31 GiB (actual), 0.6 GiB (estimated), difference: 0.29 GiB (95.2%).
(EngineCore pid=216096) INFO 09-14 19:03:32 [gpu_worker.py:860] Free memory on device (13.52/15.47 GiB) on startup. Desired GPU memory utilization is (0.35, 5.41 GiB). Actual usage is 1.6 GiB for consumed memory (weights + non-torch), 3.16 GiB for peak activation, and 0.31 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=223948493` (0.21 GiB) to fit into requested memory, or `--kv-cache-memory=8926077440` (8.31 GiB) to fully utilize gpu memory. Current kv cache memory in use is 0.66 GiB.
(EngineCore pid=216096) INFO 09-14 19:03:40 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=216096) INFO 09-14 1

## Out of domain

The draft is trained on ML papers; Docling's real workload is broader. The same measurement on
`data/ood/images` -- tax forms, court opinions, standards, central-bank minutes, lecture notes --
answers the question the in-domain number cannot: whether the acceptance survives contact with
documents the draft has never seen.

Read the per-category table, not just the aggregate. Acceptance and wall-clock can move in
*opposite* directions here: a drafter pays its per-round cost on every round, so a long-output
document can accept more tokens per step and still finish slower than plain decoding.

In [ ]:
OOD_IMAGES = ROOT / "data/ood/images"
OOD_PAGES_PER_DOC = 2      # per document; 8 categories x ~1-2 docs keeps this to ~20 pages

if "vllm" not in available():
    print("vLLM is not installed here (`uv sync --extra cuda`); skipping.")
elif not OOD_IMAGES.is_dir():
    print("no OOD corpus: run `uv run python scripts/fetch_ood.py` then\n"
          "`uv run fastdocling-prep render data/ood/docs data/ood/images`")
else:
    ood_pages = [(cat.name, page)
                 for cat in sorted(p for p in OOD_IMAGES.iterdir() if p.is_dir())
                 for doc in sorted(p for p in cat.iterdir() if p.is_dir())
                 for page in sorted(doc.glob("*.png"))[:OOD_PAGES_PER_DOC]]
    category_of = {str(page): cat for cat, page in ood_pages}

    ood_rows = sweep_speculative([page for _, page in ood_pages], [None, SPEC_CFG],
                                 cache_dir=DECODES, batched=False,
                                 gpu_memory_utilization=VLLM_GPU_FRACTION)
    report(ood_rows, f"out of domain ({len({c for c, _ in ood_pages})} categories)")

    # Per category, because the aggregate hides the spread: long-output documents (forms) pay the
    # drafter's per-round cost over many more rounds than short ones, so a drafter that breaks
    # even on the mean can still lose badly on the documents that take the longest to decode.
    by_category = {}
    for r in ood_rows:
        by_category.setdefault(category_of[str(r["page"])], {}).setdefault(r["config"], []).append(r)
    print(f"\n{'category':16s}{'pages':>6s}{'tokens':>9s}{'plain':>9s}{'draft':>9s}{'vs plain':>10s}{'tok/step':>10s}")
    for cat in sorted(by_category):
        arms = by_category[cat]
        spec_label = next(k for k in arms if k != "baseline")
        tok, base_tps, _ = pooled(arms["baseline"])
        _, spec_tps, per_step = pooled(arms[spec_label])
        print(f"{cat:16s}{len(arms['baseline']):>6d}{tok:>9,}{base_tps:>9.1f}{spec_tps:>9.1f}"
              f"{spec_tps / base_tps:>9.2f}x{per_step:>10.3f}")